In [48]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from xgboost import XGBClassifier
import joblib

print("Loading master features CSV...")
df = pd.read_csv(Path.cwd().parent /'master_dataset_lexical_features.csv')

Loading master features CSV...


In [49]:
X = df.drop(['url', 'label'], axis=1)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Data split successful. Training shape: {X_train.shape}, Testing shape: {X_test.shape}")

Data split successful. Training shape: (468027, 15), Testing shape: (117007, 15)


In [50]:
print("Initializing XGBoost Classifier...")

model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    objective='binary:logistic',
    random_state=42,
    n_jobs=-1,
    colsample_bytree=0.6,   # ← each tree only sees 60% of features
    colsample_bylevel=0.6,  # ← each split only sees 60% of features
    subsample=0.8,          # ← each tree trains on 80% of rows
    reg_alpha=0.1,          # ← L1 regularization, forces feature diversity
    min_child_weight=5, 
)

print("Training the sequential gradient boosting pipeline...")
model.fit(X_train, y_train)
print("XGBoost training cycle complete!")

Initializing XGBoost Classifier...
Training the sequential gradient boosting pipeline...
XGBoost training cycle complete!


In [51]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=['Safe', 'Malicious'])

print("\n================ XGBOOST METRICS ================")
print(f"Target Threshold Clear State: {accuracy * 100:.2f}%")
print("-------------------------------------------------")
print("Confusion Matrix Layout:")
print(cm)
print("-------------------------------------------------")
print("Classification Matrix Specs:")
print(report)


================ XGBOOST METRICS ================
Target Threshold Clear State: 95.83%
-------------------------------------------------
Confusion Matrix Layout:
[[59425   575]
 [ 4310 52697]]
-------------------------------------------------
Classification Matrix Specs:
              precision    recall  f1-score   support

        Safe       0.93      0.99      0.96     60000
   Malicious       0.99      0.92      0.96     57007

    accuracy                           0.96    117007
   macro avg       0.96      0.96      0.96    117007
weighted avg       0.96      0.96      0.96    117007



In [52]:
xgb_model = Path.cwd().parent /'backend' /'ml_model_xgb.joblib'

joblib.dump(model, xgb_model)

['/Users/AmeyaWalekar/Desktop/Summer 2026/Phisguard - CC Project/The Project /phishguard/backend/ml_model_xgb.joblib']

In [53]:
#testing

feat_imp = pd.Series(model.feature_importances_, index=X_train.columns)
print(feat_imp.sort_values(ascending=False))

path_depth            0.807624
dot_count             0.049043
url_length            0.033714
is_url_shortener      0.030232
url_entropy           0.025540
subdomain_count       0.015219
is_ip                 0.011893
hostname_length       0.011596
brand_in_subdomain    0.003952
suspicious_tld        0.003239
has_port              0.003121
digit_ratio           0.002889
hyphen_count          0.001937
at_count              0.000000
query_count           0.000000
dtype: float32


In [56]:
import joblib

model = joblib.load(Path.cwd().parent /'backend'/"ml_model_xgb.joblib")
print("Expected features:", getattr(model, "n_features_in_", "not available"))
print("Feature names:", getattr(model, "feature_names_in_", "not available"))

Expected features: 15
Feature names: ['url_length' 'hostname_length' 'dot_count' 'hyphen_count' 'at_count'
 'query_count' 'is_ip' 'url_entropy' 'subdomain_count' 'suspicious_tld'
 'digit_ratio' 'has_port' 'path_depth' 'brand_in_subdomain'
 'is_url_shortener']


In [57]:
# testing

import pandas as pd
feat_imp = pd.Series(model.feature_importances_, index=X_train.columns)
print(feat_imp.sort_values(ascending=False))

path_depth            0.807624
dot_count             0.049043
url_length            0.033714
is_url_shortener      0.030232
url_entropy           0.025540
subdomain_count       0.015219
is_ip                 0.011893
hostname_length       0.011596
brand_in_subdomain    0.003952
suspicious_tld        0.003239
has_port              0.003121
digit_ratio           0.002889
hyphen_count          0.001937
at_count              0.000000
query_count           0.000000
dtype: float32
